In [21]:
import sys
print(sys.executable)


/Users/awilliam/Documents/cas-registration/bc_obps/.venv/bin/python


In [48]:
import tabula
import pandas as pd

In [25]:
tabula.environment_info()

Python version:
    3.12.3 (main, Jan 20 2026, 12:38:40) [Clang 17.0.0 (clang-1700.6.3.2)]
Java version:
    openjdk version "25.0.2" 2026-01-20
OpenJDK Runtime Environment Homebrew (build 25.0.2)
OpenJDK 64-Bit Server VM Homebrew (build 25.0.2, mixed mode, sharing)
tabula-py version: 2.10.0
platform: macOS-26.5.1-arm64-arm-64bit
uname:
    uname_result(system='Darwin', node='XT3HQVKHCD', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')
linux_distribution: ('Darwin', '25.5.0', '')
mac_ver: ('26.5.1', ('', '', ''), 'arm64')


In [43]:
pdf_1 = 'report_attachments_2025_2024_Verification_Report_12603448-GHD-00-00-COR-EN-0001.pdf'
pdf_2 = 'report_attachments_2025_Dunkley_Final_Verification_Statement_2024.pdf'
pdf_3 = 'report_attachments_2026_2025_BC_OBPS_Verification_Report_12603448-GHD-00-00-COR-EN-0003__1.pdf'
pdf_4 = 'report_attachments_2026_BC-OBPS_Verification_Report__Statement_Yoho_Resources_2025RY.pdf'

all_pdfs = [pdf_1, pdf_2, pdf_3, pdf_4]

In [44]:
def extract_tables_from_pdf(pdf_filepath: str):
    return tabula.read_pdf(pdf_filepath, stream=True, pages="all")

In [55]:
results = {}

def read_all_pdfs():
    for pdf in all_pdfs:
        tables = extract_tables_from_pdf(pdf)
        results.setdefault(pdf, {})["all_tables"] = tables
        print(f"Extracted {len(tables)} from {pdf}")

In [ ]:
read_all_pdfs()

In [74]:
def search_headers_for_keyword(keyword: str, tables: list[pd.DataFrame]):
    matched_tables = []
    for table in tables:
        headers = table.columns.astype(str)
        if headers.str.contains(keyword, regex=True, na=False).any():
            matched_tables.append(table)
    return matched_tables

In [75]:
def search_table_headers_for_keywords():
    for filepath in results.keys():
        # keyword = "Error"
        error_tables = search_headers_for_keyword("Error", results[filepath]["all_tables"])

        # keyword = "Opinion"
        opinion_tables = search_headers_for_keyword("Opinion", results[filepath]["all_tables"])

        # keyword = "Decision"
        decision_tables = search_headers_for_keyword("Decision", results[filepath]["all_tables"])

        results[filepath]["error_tables"] = error_tables
        results[filepath]["opinion_tables"] = opinion_tables
        results[filepath]["decision_tables"] = decision_tables

In [78]:
def display_key_table_results():
    print(f"Filepath\t\t\t\t\t\tError hits\t\tOpinion hits\t\tDecision hits\n")
    for filepath in results.keys():
        print(f"\n{filepath}\t{len(results[filepath]["error_tables"])}\t\t{len(results[filepath]["opinion_tables"])}\t\t{len(results[filepath]["decision_tables"])}\n")

In [ ]:
search_table_headers_for_keywords()
display_key_table_results()

In [86]:
def display_decision_tables():
    for filepath in results.keys():
        decision_tables = results[filepath]["decision_tables"]
        if len(decision_tables) > 0:
            print(filepath)
            for table in decision_tables:
                display(table)

In [ ]:
display_decision_tables()

In [88]:
def search_dataframe_for_string(df: pd.DataFrame, search_term: str):
    matches = df[df.astype(str).apply(lambda row: row.str.contains(search_term).any(), axis=1)]
    return matches

In [134]:
def search_decision_tables_for_marked_checkbox():
    for filepath in results.keys():
        decision_tables = results[filepath]["decision_tables"]
        if len(decision_tables) > 0:
            print(filepath)
            for dt in decision_tables:
                matches = search_dataframe_for_string(dt, "☒")
                if not matches.empty:
                    display(matches)
                    parse_decision_label_from_checkbox_row(matches)
            

In [ ]:
search_decision_tables_for_marked_checkbox()

In [133]:
def parse_decision_label_from_checkbox_row(row: pd.Series):
    determination = row["Type of"].iloc[0]
    print(determination)
    return determination